In [ ]:

from DrissionPage import ChromiumOptions, ChromiumPage
from DrissionPage.common import Actions
from datetime import datetime
from pathlib import Path
from time import sleep
from urllib.parse import quote, urlparse
from urllib.request import Request, urlopen

import importlib
import json
import os
import random
import re
import ssl
import smtplib

import numpy as np
import pandas as pd
from tqdm import tqdm

from email import encoders
from email.message import EmailMessage
from email.mime.base import MIMEBase
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

# ========== 采集参数 ==========
RUN_NOW = datetime.now()
RUN_DATE = RUN_NOW.strftime('%Y-%m-%d')
RUN_TS = RUN_NOW.strftime('%Y-%m-%d %H:%M:%S')
RUN_DIR_NAME = RUN_NOW.strftime('%Y-%m-%d_%H-%M')
OUTPUT_ROOT = Path('output') / RUN_DIR_NAME
LINKS_DIR = OUTPUT_ROOT / 'links'
POSTS_DIR = OUTPUT_ROOT / 'posts'
MEDIA_ROOT = OUTPUT_ROOT / 'media'
SUMMARY_PATH = OUTPUT_ROOT / 'summary.csv'

LINKS_PER_KEYWORD = 30      # 每个关键词最多保留 20-30 条，这里取上限 30
MIN_LINKS_PER_KEYWORD = 20  # 少于该值只提醒，不追加滚动，确保页面只滚动一次
SCROLL_TIMES_PER_KEYWORD = 1
SCROLL_DISTANCE = 1500
DETAIL_LIMIT = 20
PAGE_WAIT_RANGE = (2, 4)
SCROLL_WAIT_SECONDS = 2

for folder in (LINKS_DIR, POSTS_DIR, MEDIA_ROOT):
    folder.mkdir(parents=True, exist_ok=True)


def read_lines_to_list(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return [line.strip() for line in file]


from xhs_notify import notify_failure, send_email
def yanz():
    try:
        if 'website-login' in bro.url:  
            # or bro.url == 'https://www.xiaohongshu.com/explore'
            send_email(subject='遇到验证码！！！', body='注意！验证码！！！')
            input('出现验证码，通过以后继续：')
    except Exception:
        pass


def clean_file_name(file_name):
    return re.sub(r'[<>:/\\|?*"\n\r\t]+', '', str(file_name)).strip() or 'untitled'


def parse_count(value):
    """把 1.2万、赞 88、评论 3 等文本统一为数值。"""
    if pd.isna(value):
        return 0
    text = str(value).strip().replace(',', '').replace(' ', '').replace('\t', '')
    if not text:
        return 0
    multiplier = 1
    if '万' in text or 'w' in text.lower():
        multiplier = 10000
    text = re.sub(r'(赞|点赞|收藏|评论|评|条|万|w|W)', '', text)
    match = re.search(r'\d+(?:\.\d+)?', text)
    if not match:
        return 0
    return int(float(match.group()) * multiplier)


def note_id_from_url(url):
    try:
        parsed = urlparse(str(url))
        parts = [p for p in parsed.path.split('/') if p]
        return parts[-1] if parts else f'note_{abs(hash(str(url)))}'
    except Exception:
        return f'note_{abs(hash(str(url)))}'


def normalize_url(url):
    url = str(url or '').strip()
    if not url or url.lower() == 'nan':
        return ''
    if url.startswith('//'):
        return 'https:' + url
    return url


def normalize_link_table(df):
    df = df.copy()
    rename_map = {
        '关键词': 'keyword',
        '链接': 'url',
        '标题': 'title',
        '点赞': 'like_count',
        '作者': 'author_name',
        '作者链接': 'author_url',
        '发布日期': 'publish_time_text',
        '图链接': 'cover_url',
    }
    df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns}, inplace=True)
    for col in ['keyword', 'url', 'title', 'author_name', 'author_url', 'publish_time_text', 'cover_url']:
        if col not in df.columns:
            df[col] = ''
        df[col] = df[col].fillna('').astype(str).str.strip()
    if 'like_count' not in df.columns:
        df['like_count'] = 0
    df['like_count'] = df['like_count'].apply(parse_count)
    df['url'] = df['url'].apply(normalize_url)
    df['note_id'] = df['url'].apply(note_id_from_url)
    df = df[df['url'].ne('')].copy()
    df.drop_duplicates(subset=['note_id'], keep='first', inplace=True)
    df.sort_values('like_count', ascending=False, inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df


# 8215-8216
co = ChromiumOptions().set_paths(local_port=9209).mute(True).set_argument('--start-maximized')
# co = ChromiumOptions().auto_port().mute(True).set_argument('--start-maximized')
bro = ChromiumPage(co)
bro.set.auto_handle_alert()
bro.set.when_download_file_exists('overwrite')
ac = Actions(bro)

print(f'本次采集时间: {RUN_TS}')
print(f'输出目录: {OUTPUT_ROOT.resolve()}')


本次采集日期: 2026-05-09
输出目录: /Users/wangbin/data/jupyter/实战/爬网页/小红书/自动化工作流/xhs媒体下载/output/2026-05-09


In [ ]:
keywords = [
    '数据爬虫',
    '数据采集',
    '大作业数据',
    '毕业论文数据',
    '机器学习数据',
    '小红书爬取',
    '抖音爬取',
    '公众号爬取',
    'B站爬取',
    'YouTube爬取',
    '推特爬取',
    '新闻爬取',
    '政策数据',
]

beg = 0


def search_url(keyword):
    return f'https://www.xiaohongshu.com/search_result?keyword={quote(keyword)}&source=web_explore_feed'


def safe_ele_text(ele, selector, default=''):
    try:
        return ele.ele(selector).text.strip()
    except Exception:
        return default


def safe_ele_link(ele, selector, default=''):
    try:
        return normalize_url(ele.ele(selector).link)
    except Exception:
        return default


def extract_note_cards(keyword):
    records = []
    try:
        note_items = bro.s_ele().eles('.note-item')
    except Exception:
        return records

    for rank, note_item in enumerate(note_items, start=1):
        url = safe_ele_link(note_item, '.cover mask ld') or safe_ele_link(note_item, 'tag=a')
        if not url:
            continue

        title = safe_ele_text(note_item, '.title')
        like_text = safe_ele_text(note_item, '.count', '0')
        author_name = ''
        author_url = ''
        publish_time_text = ''
        try:
            footer = note_item.ele('.footer')
            author_name = safe_ele_text(footer, '.name')
            author_url = safe_ele_link(footer, '.author')
            publish_time_text = safe_ele_text(footer, '.time')
        except Exception:
            pass
        cover_url = ''
        try:
            cover_url = normalize_url(note_item.ele('.cover mask ld').ele('tag=img').link)
        except Exception:
            pass

        records.append({
            'keyword': keyword,
            'note_id': note_id_from_url(url),
            'url': url,
            'title': title,
            'like_count': parse_count(like_text),
            'author_name': author_name,
            'author_url': author_url,
            'publish_time_text': publish_time_text,
            'cover_url': cover_url,
            'card_rank': rank,
            'collected_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        })
    return records


def collect_links_for_keyword(keyword):
    bro.get(search_url(keyword))
    bro.wait(*PAGE_WAIT_RANGE)
    yanz()

    records = extract_note_cards(keyword)

    # 按要求只做一次向下滚动，用第二屏补充候选链接。
    for _ in range(SCROLL_TIMES_PER_KEYWORD):
        bro.scroll.down(SCROLL_DISTANCE)
        sleep(SCROLL_WAIT_SECONDS)
        yanz()
        records.extend(extract_note_cards(keyword))

    df = pd.DataFrame(records)
    if df.empty:
        notify_failure(
            '关键词链接采集为空',
            details=f'关键词：{keyword}\n搜索页：{search_url(keyword)}',
            dedupe_key=f'collect-empty|{keyword}',
        )
        return pd.DataFrame(columns=[
            'keyword', 'note_id', 'url', 'title', 'like_count', 'author_name',
            'author_url', 'publish_time_text', 'cover_url', 'card_rank', 'collected_at'
        ])

    df = normalize_link_table(df)
    df = df.head(LINKS_PER_KEYWORD).copy()
    df['keyword_link_rank'] = range(1, len(df) + 1)

    safe_keyword = clean_file_name(keyword)
    csv_path = LINKS_DIR / f'{safe_keyword}_{RUN_DATE}_links.csv'
    xlsx_path = LINKS_DIR / f'{safe_keyword}_{RUN_DATE}_links.xlsx'
    df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    df.to_excel(xlsx_path, index=False, engine='openpyxl')

    if len(df) < MIN_LINKS_PER_KEYWORD:
        print(f'{keyword}: 仅采集到 {len(df)} 条，低于 {MIN_LINKS_PER_KEYWORD} 条；已按“一次滚动”策略停止。')
    else:
        print(f'{keyword}: 采集 {len(df)} 条链接 -> {csv_path}')
    return df


all_link_frames = []
for keyword in tqdm(keywords[beg:], desc='关键词链接采集'):
    try:
        all_link_frames.append(collect_links_for_keyword(keyword))
    except Exception as e:
        print(f'{keyword}: 采集失败，已跳过。原因: {e}')
        notify_failure(
            '关键词链接采集失败',
            error=e,
            details=f'关键词：{keyword}',
            dedupe_key=f'collect-keyword|{keyword}|{type(e).__name__}',
        )

links_df = normalize_link_table(pd.concat(all_link_frames, ignore_index=True)) if all_link_frames else pd.DataFrame()
links_all_csv = LINKS_DIR / 'links_all.csv'
links_all_xlsx = LINKS_DIR / 'links_all.xlsx'
links_top_csv = LINKS_DIR / f'links_top{DETAIL_LIMIT}.csv'
links_top_xlsx = LINKS_DIR / f'links_top{DETAIL_LIMIT}.xlsx'

if not links_df.empty:
    links_df.to_csv(links_all_csv, index=False, encoding='utf-8-sig')
    links_df.to_excel(links_all_xlsx, index=False, engine='openpyxl')
    links_df.head(DETAIL_LIMIT).to_csv(links_top_csv, index=False, encoding='utf-8-sig')
    links_df.head(DETAIL_LIMIT).to_excel(links_top_xlsx, index=False, engine='openpyxl')

print(f'链接汇总: {len(links_df)} 条')
print(f'链接总表: {links_all_csv}')
print(f'热门链接 Top{DETAIL_LIMIT}: {links_top_csv}')


关键词链接采集:  20%|██        | 1/5 [00:06<00:24,  6.07s/it]

数据爬虫: 采集 30 条链接 -> output/2026-05-09/links/数据爬虫_2026-05-09_links.csv


关键词链接采集:  40%|████      | 2/5 [00:13<00:20,  6.83s/it]

数据分析: 采集 30 条链接 -> output/2026-05-09/links/数据分析_2026-05-09_links.csv


关键词链接采集:  60%|██████    | 3/5 [00:19<00:13,  6.60s/it]

数据获取: 采集 30 条链接 -> output/2026-05-09/links/数据获取_2026-05-09_links.csv


关键词链接采集:  80%|████████  | 4/5 [00:25<00:06,  6.40s/it]

大学生大作业: 采集 30 条链接 -> output/2026-05-09/links/大学生大作业_2026-05-09_links.csv


关键词链接采集: 100%|██████████| 5/5 [00:31<00:00,  6.36s/it]

机器学习分析: 采集 30 条链接 -> output/2026-05-09/links/机器学习分析_2026-05-09_links.csv
链接汇总: 148 条
链接总表: output/2026-05-09/links/links_all.csv
热门链接 Top20: output/2026-05-09/links/links_top20.csv


In [3]:

"""
合并本次采集日期的链接表，并按点赞数降序生成 Top100 深抓清单。
如果刚运行过上一单元，会直接复用内存中的 links_df；否则从 output/{RUN_DIR_NAME}/links 读取。
"""

link_files = sorted(LINKS_DIR.glob(f'*_{RUN_DATE}_links.csv'))
if 'links_df' in globals() and isinstance(links_df, pd.DataFrame) and not links_df.empty:
    merged_links_df = normalize_link_table(links_df)
elif link_files:
    merged_links_df = normalize_link_table(pd.concat((pd.read_csv(p) for p in link_files), ignore_index=True))
else:
    merged_links_df = pd.DataFrame()

if merged_links_df.empty:
    raise ValueError(f'没有找到可合并的链接表: {LINKS_DIR}')

merged_links_df.to_csv(LINKS_DIR / 'links_all.csv', index=False, encoding='utf-8-sig')
merged_links_df.to_excel(LINKS_DIR / 'links_all.xlsx', index=False, engine='openpyxl')

top_links_df = merged_links_df.head(DETAIL_LIMIT).copy()
top_links_df['detail_rank'] = range(1, len(top_links_df) + 1)
top_links_df.to_csv(LINKS_DIR / f'links_top{DETAIL_LIMIT}.csv', index=False, encoding='utf-8-sig')
top_links_df.to_excel(LINKS_DIR / f'links_top{DETAIL_LIMIT}.xlsx', index=False, engine='openpyxl')

# 兼容后续单元旧变量名，但内容只保留热门 Top100。
merged_df0 = top_links_df.copy()

print(f'本次链接总数: {len(merged_links_df)}')
print(f'将抓取详情与媒体的热门内容: {len(top_links_df)}')
print(f'链接目录: {LINKS_DIR.resolve()}')


本次链接总数: 148
将抓取详情与媒体的热门内容: 20
链接目录: /Users/wangbin/data/jupyter/实战/爬网页/小红书/自动化工作流/xhs媒体下载/output/2026-05-09/links


In [ ]:

"""
抓取热门 Top100 推文详情与媒体资源，并输出规范化结果：
output/{RUN_DIR_NAME}/links/   链接清单
output/{RUN_DIR_NAME}/posts/   详情表
output/{RUN_DIR_NAME}/media/   媒体文件，按 note_id 分目录
output/{RUN_DIR_NAME}/summary.csv  链接 + 详情 + 媒体汇总表
"""

import 视频下载 as vd
import 图片下载 as imgd
from xhs_notify import notify_failure, notify_failure_once

vd = importlib.reload(vd)
imgd = importlib.reload(imgd)

COOKIE = ''  # 如遇 403，可填浏览器 cookie
HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
        'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'
    ),
    'Referer': 'https://www.xiaohongshu.com/',
    'Accept': '*/*',
}


def infer_ext(url, default='.bin'):
    low = str(url).lower()
    if '.mp4' in low:
        return '.mp4'
    if '.webm' in low:
        return '.webm'
    if '.jpg' in low or '.jpeg' in low:
        return '.jpg'
    if '.png' in low:
        return '.png'
    if '.webp' in low:
        return '.webp'
    return default


def download_binary(url, save_path, retries=2, timeout=60):
    save_path = Path(save_path)
    headers = dict(HEADERS)
    if COOKIE.strip():
        headers['Cookie'] = COOKIE.strip()

    tmp = save_path.with_suffix(save_path.suffix + '.part')
    for attempt in range(retries + 1):
        try:
            req = Request(url=url, headers=headers)
            with urlopen(req, timeout=timeout) as resp, tmp.open('wb') as f:
                while True:
                    chunk = resp.read(1024 * 1024)
                    if not chunk:
                        break
                    f.write(chunk)
            if tmp.exists() and tmp.stat().st_size > 0:
                tmp.replace(save_path)
                return True
        except Exception as e:
            if tmp.exists():
                try:
                    tmp.unlink()
                except Exception:
                    pass
            if attempt >= retries:
                print(f'媒体下载失败: {url} -> {save_path.name}，原因: {e}')
                notify_failure(
                    '详情媒体下载失败',
                    error=e,
                    details=f'URL: {url}\n保存路径: {save_path}',
                    dedupe_key=f'detail-media|{url}',
                )
            else:
                sleep(1 + attempt)
    return False


def open_note_page(url, retries=2):
    global bro, ac
    last_error = None
    for attempt in range(retries + 1):
        try:
            yanz()
            bro.get(url)
            bro.wait(*PAGE_WAIT_RANGE)
            yanz()
            return bro.s_ele()
        except Exception as e:
            last_error = e
            sleep(2 + attempt)
            try:
                bro = ChromiumPage(co)
                bro.set.auto_handle_alert()
                bro.set.when_download_file_exists('overwrite')
                ac = Actions(bro)
            except Exception:
                pass
    raise last_error


def get_script_text():
    chunks = []
    try:
        for script in bro.eles('tag=script'):
            html = script.html or ''
            if html:
                chunks.append(html)
    except Exception:
        pass
    if chunks:
        return '\n'.join(chunks)
    try:
        return bro.ele('tag=script', index=-6).html or ''
    except Exception:
        return ''


def extract_media_urls(script_text):
    video_urls = []
    image_urls = []
    if not script_text:
        return video_urls, image_urls

    try:
        video_urls = vd.extract_video_urls(script_text, quality='highest').video_urls
    except Exception as e:
        video_urls = []
        notify_failure_once('视频链接解析失败', error=e, details='详情页 script 解析视频 URL 失败')

    if not video_urls:
        try:
            image_urls = imgd.extract_image_urls(
                raw_text=script_text,
                quality='highest',
                resolution_mode='best',
            ).image_urls
        except Exception as e:
            image_urls = []
            notify_failure_once('图片链接解析失败', error=e, details='详情页 script 解析图片 URL 失败')
    return video_urls, image_urls


def extract_detail_record(row, idx, total):
    url = row['url']
    note_id = row.get('note_id') or note_id_from_url(url)
    note_dir = MEDIA_ROOT / note_id
    record = {
        'note_id': note_id,
        'url': url,
        'note_text': '',
        'topics': '',
        'collect_count': 0,
        'comment_count': 0,
        'media_type': 'unknown',
        'media_dir': '',
        'media_file_count': 0,
        'media_files': '[]',
        'detail_status': 'ok',
        'detail_error': '',
        'fetched_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    }

    try:
        s_bro = open_note_page(url)
    except Exception as e:
        record['detail_status'] = 'open_failed'
        record['detail_error'] = str(e)
        print(f'[{idx}/{total}] {note_id} 打开失败，跳过详情。')
        notify_failure(
            '推文详情打开失败',
            error=e,
            details=f'note_id: {note_id}\nURL: {url}',
            dedupe_key=f'open-detail|{note_id}',
        )
        return record

    try:
        note_text = s_bro.ele('#detail-desc').text.strip()
        record['note_text'] = note_text
        topics = re.findall(r'#([\w\u4e00-\u9fff-]+)', note_text)
        record['topics'] = ' '.join(dict.fromkeys(topics))
    except Exception:
        pass

    try:
        record['collect_count'] = parse_count(s_bro.ele('.collect-wrapper').text)
    except Exception:
        pass

    try:
        record['comment_count'] = parse_count(s_bro.ele('.chat-wrapper').text)
    except Exception:
        pass

    video_urls, image_urls = extract_media_urls(get_script_text())
    media_paths = []

    if not video_urls and not image_urls:
        record['detail_status'] = 'no_media_found'
        record['detail_error'] = '未解析到视频或图片链接'
        notify_failure(
            '推文媒体解析为空',
            details=f'note_id: {note_id}\nURL: {url}',
            dedupe_key=f'no-media|{note_id}',
        )

    if video_urls:
        record['media_type'] = 'video'
        note_dir.mkdir(parents=True, exist_ok=True)
        video_path = note_dir / f'{note_id}.mp4'
        if download_binary(video_urls[0], video_path):
            media_paths.append(str(video_path))
    elif image_urls:
        record['media_type'] = 'image'
        note_dir.mkdir(parents=True, exist_ok=True)
        for media_idx, image_url in enumerate(image_urls, start=1):
            ext = infer_ext(image_url, default='.jpg')
            image_path = note_dir / f'{note_id}_{media_idx:02d}{ext}'
            if download_binary(image_url, image_path):
                media_paths.append(str(image_path))

    if media_paths:
        record['media_dir'] = str(note_dir)
        record['media_file_count'] = len(media_paths)
        record['media_files'] = json.dumps(media_paths, ensure_ascii=False)

    print(f"[{idx}/{total}] {note_id} | {record['media_type']} | 下载{record['media_file_count']}个 | {record['detail_status']}")
    return record


links_top_path = LINKS_DIR / f'links_top{DETAIL_LIMIT}.csv'
if links_top_path.exists():
    top_links_df = normalize_link_table(pd.read_csv(links_top_path)).head(DETAIL_LIMIT)
elif 'merged_df0' in globals() and isinstance(merged_df0, pd.DataFrame) and not merged_df0.empty:
    top_links_df = normalize_link_table(merged_df0).head(DETAIL_LIMIT)
else:
    top_links_df = normalize_link_table(pd.read_csv(LINKS_DIR / 'links_all.csv')).head(DETAIL_LIMIT)

detail_records = []
for idx, (_, row) in enumerate(top_links_df.iterrows(), start=1):
    detail_records.append(extract_detail_record(row, idx, len(top_links_df)))

    # 每条写一次，长时间运行时中断也能保留进度。
    posts_df = pd.DataFrame(detail_records)
    posts_df.to_csv(POSTS_DIR / f'posts_top{DETAIL_LIMIT}.csv', index=False, encoding='utf-8-sig')
    posts_df.to_excel(POSTS_DIR / f'posts_top{DETAIL_LIMIT}.xlsx', index=False, engine='openpyxl')

    summary_df = pd.merge(top_links_df, posts_df, on=['note_id', 'url'], how='left')
    summary_df.to_csv(SUMMARY_PATH, index=False, encoding='utf-8-sig')

print(f'详情表: {(POSTS_DIR / f"posts_top{DETAIL_LIMIT}.csv").resolve()}')
print(f'汇总表: {SUMMARY_PATH.resolve()}')
print(f'媒体目录: {MEDIA_ROOT.resolve()}')


[1/20] 673f3206000000000203af6d | image | 下载8个 | ok
[2/20] 674c0bc9000000000702b812 | image | 下载10个 | ok
[3/20] 681d6bed0000000022029100 | image | 下载14个 | ok
[4/20] 69b8dae6000000001f0020c0 | image | 下载2个 | ok
[5/20] 69e7294200000000220032ec | image | 下载36个 | ok
[6/20] 68ce62fa000000000e02077e | image | 下载12个 | ok
[7/20] 69e37b810000000023023504 | image | 下载2个 | ok
[8/20] 69c9ba90000000001f00613b | image | 下载4个 | ok
[9/20] 68becaf8000000001d00e5e3 | image | 下载4个 | ok
[10/20] 68af2c89000000001b036fd9 | image | 下载20个 | ok
[11/20] 68f0c6a3000000000402a530 | image | 下载26个 | ok
[12/20] 67a22409000000001800cdd7 | image | 下载28个 | ok
[13/20] 67e77a64000000001d0395dc | image | 下载2个 | ok
[14/20] 695155550000000022008e13 | video | 下载1个 | ok
[15/20] 67c651c7000000002900a315 | image | 下载2个 | ok
[16/20] 69a80148000000002202f020 | image | 下载2个 | ok


In [ ]:

"""
本次采集结果预览。
核心结果已经由前面的流程输出到：
- output/{RUN_DIR_NAME}/links/
- output/{RUN_DIR_NAME}/posts/
- output/{RUN_DIR_NAME}/media/
- output/{RUN_DIR_NAME}/summary.csv
"""

if SUMMARY_PATH.exists():
    summary_df = pd.read_csv(SUMMARY_PATH)
    print(f'汇总记录数: {len(summary_df)}')
    print(f'汇总表路径: {SUMMARY_PATH.resolve()}')
    display(summary_df.head(10))
else:
    print(f'还没有生成 summary.csv，请先运行链接采集、合并和详情抓取单元: {SUMMARY_PATH}')


## 模块化视频下载（支持最高/最低清晰度）
下面这个单元会按你的流程执行：
- `bro.listen.start(url)`
- `bro.get(url)`
- 自动提取视频链接并下载

默认使用 `quality="highest"` 下载最高质量视频；只有明确需要省流量时才改成 `"lowest"`。

In [ ]:
import 视频下载 as vd

# 你的输入 URL
url = 'https://www.xiaohongshu.com/search_result/69ad4dc4000000002603082e?xsec_token=ABf4dXXbEDWc2MbwuorvBHW5Zd-t4yDQQSYrV1lmLUxrU=&xsec_source='

# 默认最高质量；如需省流量可改成 "lowest"
quality = "highest"

# 会自动执行：bro.listen.start(url) + bro.get(url)
result = vd.download_from_page(
    bro=bro,
    url=url,
    output_dir=str(MEDIA_ROOT / 'demo_video_downloads'),
    quality=quality,
    start_listen=True,
    script_index=-6,
    cookie=''  # 如遇 403，可粘贴 cookie
)

print('提取到视频数:', len(result['extract'].video_urls))
print('下载成功数:', result['download'].success)
print('下载目录:', result['download'].output_dir)

# 如果你已经在前面单元拿到了 ele（script 文本），也可以直接这样调用：
# result2 = vd.download_from_text(ele, output_dir=str(MEDIA_ROOT / 'demo_video_downloads'), quality='highest')


## 模块化图片下载（默认最高质量）
下面这个单元会按你的流程执行：
- `bro.listen.start(url)`
- `bro.get(url)`
- 默认按每组图片选择最高质量链接
- 下载结果进入 `high_resolution` 目录

可配置：
- `quality="highest"`：保留最高质量
- `quality="lowest"`：保留最低质量
- `quality="none"`：保留候选用于人工比较
- `resolution_mode`：默认 `"best"`；调试候选时再用 `"all"`

In [ ]:
import importlib
import inspect
import 图片下载 as imgd

# 强制重载，避免 Notebook 复用旧模块导致参数不匹配
imgd = importlib.reload(imgd)
print('图片模块路径:', imgd.__file__)
print('download_from_page 签名:', inspect.signature(imgd.download_from_page))

# 你的输入 URL（图片帖子）
url = 'https://www.xiaohongshu.com/explore/69ad4dc4000000002603082e?xsec_token=ABf4dXXbEDWc2MbwuorvBHW5Zd-t4yDQQSYrV1lmLUxrU=&xsec_source='

# 默认最高质量；如需人工比较候选可改成 "none"
quality = "highest"

# 默认："best"（每组只取最高质量）；调试候选时可改成 "all"
resolution_mode = "best"

# 会自动执行：bro.listen.start(url) + bro.get(url)
result_img = imgd.download_from_page(
    bro=bro,
    url=url,
    output_dir=str(MEDIA_ROOT / 'demo_image_downloads'),
    quality=quality,
    resolution_mode=resolution_mode,
    start_listen=True,
    script_index=-6,
    cookie=''  # 如遇 403，可粘贴 cookie
)

print('提取到有效图片数:', len(result_img['extract'].image_urls))
print('下载成功数(最终保留):', result_img['download'].success)
print('原始下载成功数:', result_img['download'].downloaded_success)
print('未保留候选数:', result_img['download'].removed_by_quality)
print('图片组数:', result_img['download'].group_count)
print('高分目录文件数:', result_img['download'].high_count)
print('低分目录文件数:', result_img['download'].low_count)
print('高分目录:', result_img['download'].high_dir)
print('低分目录:', result_img['download'].low_dir)
print('下载目录:', result_img['download'].output_dir)
print('分辨率模式:', result_img['extract'].resolution_mode)

# 如果你已经在前面单元拿到了 ele（script 文本），也可以直接这样调用：
# result_img2 = imgd.download_from_text(
#     ele,
#     output_dir=str(MEDIA_ROOT / 'demo_image_downloads'),
#     quality='highest',
#     resolution_mode='best'
# )
